# Ajustes NOGSE libres con lóbulos rectangulares (`M_nogse_free`)

Notebook para el fantoma de agua (set v2.00, $128\times128$). Lee `master.long.parquet`, ajusta solo $D_0$ (con $M_0=1$) a cada señal Hahn y CPMG y al contraste CPMG − Hahn, y guarda figuras y tablas en `signal_analysis/fittings/rectangular_lobes/`.

Modelo: **lóbulos rectangulares**, los modelos NOGSE de `src/models/model_fitting.py` (`M_nogse_free`). Las rampas de 1.2 ms se ignoran: cada lóbulo es un rectángulo de amplitud $G$ y duración $x/2$ o $y/2$. El notebook análogo con el $b$ exacto de los trapecios es `nogse_fits_trapezoidal_lobes.ipynb`; los dos escriben las mismas figuras y tablas en carpetas distintas para compararlas.

## Qué datos entran y de dónde sale cada parámetro

Cada fila de `master.long.parquet` es la señal de un volumen en un ROI. Este notebook usa solo `roi = water`, `stat = avg` y las series CPMG y Hahn (la serie b0 PA es solo para TOPUP).

| Columna | Símbolo | Qué es | De dónde viene |
|---|---|---|---|
| `value` | $S(G)$ | media de la señal en el ROI de agua (máscara manual) del volumen | `signal_extraction` sobre el NIfTI (`raw` o `den_gr`) |
| `S0`, `value_norm` | $S_0$, $M = S/S_0$ | $S_0$ es la media de los dos volúmenes b0 de la **misma** serie; $M(G=0)=1$ por construcción | ingest de `signal_analysis` |
| `TN` | $T$ | duración total del bloque NOGSE, $T_{NOGSE}$ (ms); también `td_ms` | planilla `sequence_parameters_phantoms.xlsx` = protocolo YAML del `.seq` |
| `N` | $N$ | número de períodos de la modulación (aquí $N=2$) | ídem |
| `x` | $x$ | período CPMG (ms): $x=T/N$ en la serie CPMG; $x = 5$ ms (fijo) en la serie Hahn | ídem |
| `y` | $y$ | período Hahn, $y = T-(N-1)x$ (ms) | ídem |
| `ramp_time_ms` | $t_r$ | tiempo de rampa de cada trapecio, 1.2 ms | ídem |
| `bvalue` | $b$ | b-value del `.bval` (s/mm²): traza de la matriz $b$ **exacta** de los trapecios jugados, calculada por el generador de la secuencia | `.bval` de BIDS |
| `g` | $G$ | amplitud del gradiente NOGSE (mT/m): 0, 7, …, 70 equiespaciado | ingest: $G=\sqrt{b/k_{trap}(T,N,x,t_r)}$, que invierte exactamente el `.bval` |
| `direction` | — | eje del gradiente en el escáner: 1 = $x$, 2 = $y$, 3 = $z$ (bvec $[1,0,0]$, $[0,1,0]$, $[0,0,1]$) | `.bvec` |

**Forma de onda efectiva** (signo invertido por los pulsos π virtuales, que en esta secuencia Pulseq se implementan con la polaridad del gradiente): para $N=2$ son cuatro lóbulos de amplitud $G$ y signos $+,-,-,+$ con duraciones $x/2,\ x/2,\ y/2,\ y/2$. El vector de onda es $q(t)=\gamma\int_0^t g(t')\,dt'$ y vuelve a cero al final del bloque. Para difusión libre con coeficiente $D_0$,

$$M = \frac{S}{S_0} = M_0\,e^{-b\,D_0},\qquad b=\int_0^{T} q(t)^2\,dt = k(T,N,x)\,G^2 .$$

Todo depende de $G^2$: el factor $k$ (s/mm² por (mT/m)²) solo depende de la temporización. **La única diferencia entre los dos notebooks es cómo se calcula $k$** (lóbulos rectangulares o trapezoidales); el resto de las celdas es idéntico.

**Ajuste.** Para cada curva (tipo, $T$, dirección) se ajusta un único parámetro, $D_0$, con $M_0=1$ fijo (porque $M(0)=1$), minimizando el error cuadrático medio en escala lineal sobre los 11 puntos ($G=0,\dots,70$ mT/m). Se optimiza $\ln D_0$ en $[10^{-4},10^{-2}]$ mm²/s para mantener $D_0>0$.

**Contraste.** $\Delta M(G) = M_{CPMG}(G) - M_{Hahn}(G) = e^{-k_{CPMG}G^2D_0} - e^{-k_{Hahn}G^2D_0}$, con los puntos emparejados por $T$, dirección y `b_step` (mismo $G$). Se ajusta también un solo $D_0$ por ($T$, dirección). Con un Hahn ideal ($x\to0$) y lóbulos rectangulares, $k_{CPMG}/k_{Hahn}=1/N^2=1/4$ para todo $T$ y el pico del contraste vale $0.4725$ siempre; con $x_{Hahn}=5$ ms fijo el cociente cambia con $T$ y el pico también.

**Advertencia.** Con $S/S_0 \lesssim 0.2$ los puntos quedan sobre el piso de ruido de la magnitud (≈ 0.11 en este fantoma) y el modelo exponencial no lo describe; los $D_0$ de $T$ largos están sesgados por eso.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import minimize_scalar


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'Data-BIDS').is_dir() and (candidate / 'repos' / 'signal_analysis').is_dir():
            return candidate
    raise FileNotFoundError('No encuentro la raíz del proyecto.')


PROJECT_ROOT = find_project_root()
REPO_ROOT = PROJECT_ROOT / 'repos' / 'signal_analysis'
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from models.model_fitting import _GYRO, M_nogse_free

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)
PROJECT_ROOT

## Configuración

Por defecto se usa la rama denoised + Gibbs-removal (`den_gr--manual`). Cambiar solo `ANALYSIS_TAG` permite repetir exactamente el análisis con `raw--manual`. `MODEL_NAME` define la carpeta de salida; el otro notebook escribe los mismos archivos en `OTHER_MODEL_NAME`.

In [ ]:
MODEL_NAME = 'rectangular_lobes'
OTHER_MODEL_NAME = 'trapezoidal_lobes'
ANALYSIS_TAG = 'den_gr--manual'  # alternativa: 'raw--manual'
ROI = 'water'
STAT = 'avg'
D0_BOUNDS_MM2_S = (1e-4, 1e-2)
MM2_S_TO_M2_MS = 1e-9  # D0 de model_fitting en m^2/ms

MASTER_PATH = (PROJECT_ROOT / 'Data-BIDS' / 'derivatives' / 'signal_analysis' /
               ANALYSIS_TAG / 'phantoms' / 'nogse_experiments' / 'master.long.parquet')
OUTPUT_ROOT = REPO_ROOT / 'fittings' / MODEL_NAME
SIGNAL_DIR = OUTPUT_ROOT / 'signals'
CONTRAST_DIRECTION_DIR = OUTPUT_ROOT / 'contrasts_by_direction'
CONTRAST_TIME_DIR = OUTPUT_ROOT / 'contrasts_by_time'
TABLE_DIR = OUTPUT_ROOT / 'tables'
for folder in (SIGNAL_DIR, CONTRAST_DIRECTION_DIR, CONTRAST_TIME_DIR, TABLE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

master = pd.read_parquet(MASTER_PATH)
data = master.loc[(master['stat'].astype(str) == STAT) &
                  (master['roi'].astype(str) == ROI) &
                  (master['type'].astype(str).str.upper().isin(['CPMG', 'HAHN']))].copy()
data['type'] = data['type'].astype(str).str.upper()
data['direction'] = data['direction'].astype(str)
for column in ['td_ms', 'TN', 'N', 'x', 'ramp_time_ms', 'g', 'bvalue', 'b_step', 'value_norm']:
    data[column] = pd.to_numeric(data[column], errors='coerce')
data = data.dropna(subset=['td_ms', 'TN', 'N', 'x', 'ramp_time_ms', 'g', 'bvalue', 'b_step', 'value_norm'])
if not np.allclose(data['TN'], data['td_ms']):
    raise ValueError('TN y td_ms no coinciden: revisar la tabla de parámetros.')

print('Modelo:', MODEL_NAME)
print('Master:', MASTER_PATH)
print('Salida:', OUTPUT_ROOT)
display(data.groupby(['type', 'td_ms', 'direction']).size().rename('n').reset_index())

## Modelo: lóbulos rectangulares

Con lóbulos rectangulares $q(t)$ es una función triangular: en cada período CPMG $x$ (lóbulos $+$ y $-$ de duración $x/2$) sube linealmente hasta $\gamma G x/2$ y vuelve a cero, y lo mismo en el período Hahn $y$. Cada período aporta

$$\int_{\text{período } p} q^2\,dt = 2\int_0^{p/2}(\gamma G t)^2\,dt = \frac{\gamma^2G^2p^3}{12},$$

y como hay $N-1$ períodos CPMG y uno Hahn,

$$k_{rect}(T,N,x) = \frac{\gamma^2}{12}\left[(N-1)x^3 + y^3\right],\qquad y = T-(N-1)x .$$

Es exactamente `M_nogse_free(TE, G, N, x, M0, D0)` de `model_fitting.py`:

$$M = M_0\exp\!\left(-\frac{\gamma^2 G^2 D_0}{12}\left[(N-1)x^3+y^3\right]\right),$$

con `TE` $=T$ y $x$ en ms, $G$ en mT/m, $\gamma$ = `_GYRO` = 267.522 rad ms⁻¹ mT⁻¹ y $D_0$ en m²/ms ($1\ \mathrm{mm^2/s}=10^{-9}\ \mathrm{m^2/ms}$). En s/mm² por (mT/m)²: $k_{rect} = \gamma^2[(N-1)x^3+y^3]/12/10^9$.

Para el contraste no se usa `NOGSE_contrast_vs_g_free`, porque fija $x_{Hahn}=0$: aquí se resta `M_nogse_free` con $x=T/N$ menos `M_nogse_free` con $x=5$ ms.

**Qué ignora:** las rampas. En la secuencia cada lóbulo de duración $d$ es un trapecio (rampa $t_r$, meseta $d-2t_r$, rampa $t_r$) **dentro** del mismo período, así que tiene menos área que el rectángulo y el $b$ real es menor: $k_{trap}<k_{rect}$. Con el mismo dato, el $D_0$ ajustado aquí sale menor que con el $b$ exacto en el factor $k_{trap}/k_{rect}$ (tabla de la última sección).

In [ ]:
def k_model(tn_ms, n_value, x_ms, ramp_ms):
    # s/mm^2 per (mT/m)^2 of M_nogse_free (rectangular lobes; ramp_ms is not used)
    y_ms = tn_ms - (n_value - 1) * x_ms
    return _GYRO**2 * ((n_value - 1) * x_ms**3 + y_ms**3) / 12.0 / 1e9


def signal_model(tn_ms, g_mT_m, n_value, x_ms, ramp_ms, d0_mm2_s):
    return M_nogse_free(tn_ms, np.asarray(g_mT_m, float), n_value, x_ms, 1.0, d0_mm2_s * MM2_S_TO_M2_MS)

Funciones comunes a los dos notebooks: el modelo de contraste, el ajuste de un parámetro y la lectura de $T$, $N$, $x$, $t_r$ de cada curva.

In [ ]:
def contrast_model(tn_ms, g_mT_m, n_value, x_cpmg_ms, x_hahn_ms, ramp_ms, d0_mm2_s):
    return (signal_model(tn_ms, g_mT_m, n_value, x_cpmg_ms, ramp_ms, d0_mm2_s)
            - signal_model(tn_ms, g_mT_m, n_value, x_hahn_ms, ramp_ms, d0_mm2_s))


def fit_one_parameter(y, model):
    y = np.asarray(y, float)
    valid = np.isfinite(y)
    lo, hi = np.log(D0_BOUNDS_MM2_S)

    def loss(log_d0):
        prediction = np.asarray(model(np.exp(log_d0)), float)
        return np.mean((y[valid] - prediction[valid]) ** 2)

    result = minimize_scalar(loss, bounds=(lo, hi), method='bounded')
    d0 = float(np.exp(result.x))
    prediction = np.asarray(model(d0), float)
    residual = y[valid] - prediction[valid]
    return {
        'D0_mm2_s': d0,
        'rmse': float(np.sqrt(np.mean(residual**2))),
        'n_points': int(valid.sum()),
        'ok': bool(result.success),
    }


def timing(curve):
    row = curve.iloc[0]
    return float(row['TN']), float(row['N']), float(row['x']), float(row['ramp_time_ms'])


def d0_label(d0):
    return f'$D_0$={d0 * 1e3:.3f} × 10$^{{-3}}$ mm²/s'


def save_figure(fig, folder, stem):
    fig.savefig(folder / f'{stem}.png', dpi=180, bbox_inches='tight')
    fig.savefig(folder / f'{stem}.pdf', bbox_inches='tight')

## Hahn y CPMG: una figura por medición

Cada figura contiene las tres direcciones y su propio $D_0$ ajustado. Se generan ocho figuras: Hahn y CPMG para cada uno de los cuatro tiempos.

In [ ]:
direction_colors = {'1': 'tab:blue', '2': 'tab:orange', '3': 'tab:green'}
signal_fit_rows = []

for (sequence_type, td_ms), measurement in data.groupby(['type', 'td_ms'], sort=True):
    fig, ax = plt.subplots(figsize=(7.2, 5.0))
    for direction, curve in measurement.groupby('direction', sort=True):
        curve = curve.sort_values('g')
        tn_ms, n_value, x_ms, ramp_ms = timing(curve)
        g = curve['g'].to_numpy(float)
        y = curve['value_norm'].to_numpy(float)
        fit = fit_one_parameter(y, lambda d0: signal_model(tn_ms, g, n_value, x_ms, ramp_ms, d0))
        signal_fit_rows.append({
            'model': MODEL_NAME, 'analysis_tag': ANALYSIS_TAG, 'roi': ROI, 'type': sequence_type,
            'td_ms': td_ms, 'direction': direction, 'N': n_value, 'x_ms': x_ms,
            'k_s_mm2_per_mT2_m2': k_model(tn_ms, n_value, x_ms, ramp_ms), **fit
        })
        color = direction_colors.get(direction)
        ax.plot(g, y, 'o', color=color, ms=5)
        g_fine = np.linspace(0, g.max(), 400)
        ax.plot(g_fine, signal_model(tn_ms, g_fine, n_value, x_ms, ramp_ms, fit['D0_mm2_s']),
                color=color, lw=2, label=f'dir {direction}: {d0_label(fit["D0_mm2_s"])}')
    ax.set(title=f'{sequence_type} — $T_D$={td_ms:g} ms ({MODEL_NAME})', xlabel='G (mT/m)',
           ylabel='Señal normalizada', xlim=(0, None), ylim=(-0.03, 1.04))
    ax.legend(fontsize=9)
    save_figure(fig, SIGNAL_DIR, f'signal_{sequence_type}_Td-{td_ms:g}ms')
    plt.show()

signal_fits = pd.DataFrame(signal_fit_rows).sort_values(['type', 'td_ms', 'direction'])
signal_fits.to_csv(TABLE_DIR / 'signal_fits.csv', index=False)
display(signal_fits)

## Construcción y ajuste del contraste CPMG − Hahn

Las curvas se emparejan estrictamente por tiempo, dirección y `b_step`. También se verifica que $G$, $N$, $T$ y $t_r$ coincidan. Cada combinación tiempo-dirección tiene un único ajuste de $D_0$, reutilizado en las dos galerías siguientes.

In [ ]:
keys = ['td_ms', 'direction', 'b_step']
paired = ['g', 'bvalue', 'TN', 'N', 'x', 'ramp_time_ms', 'value_norm']
cpmg = data.loc[data['type'].eq('CPMG'), keys + paired].rename(columns={c: f'{c}_cpmg' for c in paired})
hahn = data.loc[data['type'].eq('HAHN'), keys + paired].rename(columns={c: f'{c}_hahn' for c in paired})
contrast = cpmg.merge(hahn, on=keys, how='inner', validate='one_to_one')
for column in ['g', 'TN', 'N', 'ramp_time_ms']:
    if not np.allclose(contrast[f'{column}_cpmg'], contrast[f'{column}_hahn'], rtol=1e-6, atol=1e-6):
        raise ValueError(f'CPMG y Hahn no tienen el mismo {column} para todos los puntos emparejados.')
contrast['g'] = contrast['g_cpmg']
contrast['value_contrast'] = contrast['value_norm_cpmg'] - contrast['value_norm_hahn']


def contrast_timing(curve):
    row = curve.iloc[0]
    return (float(row['TN_cpmg']), float(row['N_cpmg']), float(row['x_cpmg']), float(row['x_hahn']),
            float(row['ramp_time_ms_cpmg']))


contrast_fit_rows = []
for (td_ms, direction), curve in contrast.groupby(['td_ms', 'direction'], sort=True):
    curve = curve.sort_values('g')
    tn_ms, n_value, x_cpmg, x_hahn, ramp_ms = contrast_timing(curve)
    g = curve['g'].to_numpy(float)
    y = curve['value_contrast'].to_numpy(float)
    fit = fit_one_parameter(y, lambda d0: contrast_model(tn_ms, g, n_value, x_cpmg, x_hahn, ramp_ms, d0))
    contrast_fit_rows.append({
        'model': MODEL_NAME, 'analysis_tag': ANALYSIS_TAG, 'roi': ROI, 'td_ms': td_ms, 'direction': direction,
        'N': n_value, 'x_cpmg_ms': x_cpmg, 'x_hahn_ms': x_hahn,
        'k_ratio_cpmg_over_hahn': k_model(tn_ms, n_value, x_cpmg, ramp_ms) / k_model(tn_ms, n_value, x_hahn, ramp_ms), **fit
    })

contrast_fits = pd.DataFrame(contrast_fit_rows).sort_values(['td_ms', 'direction'])
contrast_fits.to_csv(TABLE_DIR / 'contrast_fits.csv', index=False)
display(contrast_fits)

## Contrastes: cuatro tiempos para cada dirección

Cada figura tiene exactamente cuatro curvas, una para cada tiempo disponible.

In [ ]:
time_colors = dict(zip(sorted(contrast['td_ms'].unique()), plt.cm.viridis(np.linspace(0.1, 0.9, contrast['td_ms'].nunique()))))


def plot_contrast_curve(ax, curve, d0, color, label):
    curve = curve.sort_values('g')
    tn_ms, n_value, x_cpmg, x_hahn, ramp_ms = contrast_timing(curve)
    ax.plot(curve['g'], curve['value_contrast'], 'o', color=color, ms=5)
    g_fine = np.linspace(0, curve['g'].max(), 400)
    ax.plot(g_fine, contrast_model(tn_ms, g_fine, n_value, x_cpmg, x_hahn, ramp_ms, d0), color=color, lw=2,
            label=f'{label}: {d0_label(d0)}')


def contrast_d0(td_ms, direction):
    return contrast_fits.loc[contrast_fits['td_ms'].eq(td_ms) & contrast_fits['direction'].eq(direction), 'D0_mm2_s'].iloc[0]


for direction, panel in contrast.groupby('direction', sort=True):
    fig, ax = plt.subplots(figsize=(7.2, 5.0))
    for td_ms, curve in panel.groupby('td_ms', sort=True):
        plot_contrast_curve(ax, curve, contrast_d0(td_ms, direction), time_colors[td_ms], f'$T_D$={td_ms:g} ms')
    ax.set(title=f'Contraste CPMG − Hahn — dirección {direction} ({MODEL_NAME})', xlabel='G (mT/m)',
           ylabel='Contraste normalizado', xlim=(0, None))
    ax.legend(fontsize=8.5)
    save_figure(fig, CONTRAST_DIRECTION_DIR, f'contrast_direction-{direction}')
    plt.show()

## Contrastes: mismo tiempo y distinta dirección

Esta galería muestra directamente cuánto coinciden (o no) las tres direcciones a cada tiempo.

In [ ]:
for td_ms, panel in contrast.groupby('td_ms', sort=True):
    fig, ax = plt.subplots(figsize=(7.2, 5.0))
    for direction, curve in panel.groupby('direction', sort=True):
        plot_contrast_curve(ax, curve, contrast_d0(td_ms, direction), direction_colors.get(direction), f'dir {direction}')
    ax.set(title=f'Contraste CPMG − Hahn — $T_D$={td_ms:g} ms ({MODEL_NAME})', xlabel='G (mT/m)',
           ylabel='Contraste normalizado', xlim=(0, None))
    ax.legend(fontsize=9)
    save_figure(fig, CONTRAST_TIME_DIR, f'contrast_Td-{td_ms:g}ms')
    plt.show()

## Diagnóstico de las diferencias entre direcciones

`max_direction_range` es la máxima separación absoluta entre las tres curvas medidas a un mismo $G$. `rms_direction_spread` resume la dispersión respecto de la media direccional. Estos valores se calculan antes del ajuste y son iguales en los dos notebooks. `D0_cv_percent` cuantifica la variación de los $D_0$ ajustados entre direcciones.

In [ ]:
qc_rows = []
for (sequence_type, td_ms), panel in data.groupby(['type', 'td_ms'], sort=True):
    wide = panel.pivot(index='b_step', columns='direction', values='value_norm')
    point_ranges = wide.max(axis=1) - wide.min(axis=1)
    point_spread = wide.sub(wide.mean(axis=1), axis=0)
    fitted = signal_fits.loc[(signal_fits['type'].eq(sequence_type)) & signal_fits['td_ms'].eq(td_ms), 'D0_mm2_s']
    qc_rows.append({
        'model': MODEL_NAME, 'type': sequence_type, 'td_ms': td_ms,
        'max_direction_range': float(point_ranges.max()),
        'rms_direction_spread': float(np.sqrt(np.nanmean(point_spread.to_numpy()**2))),
        'D0_mean_mm2_s': float(fitted.mean()),
        'D0_cv_percent': float(100 * fitted.std(ddof=1) / fitted.mean()),
    })

direction_qc = pd.DataFrame(qc_rows)
direction_qc.to_csv(TABLE_DIR / 'direction_qc.csv', index=False)
display(direction_qc)

## Comprobación: $k$ del modelo frente al `bvalue` de la tabla

`k_model` es el $k$ de este notebook; `k_from_bvalue` = mediana de `bvalue`/$G^2$ (el $b$ exacto de la secuencia). En el notebook trapezoidal el cociente es 1 (salvo $3\times10^{-5}$ por $\gamma$); en el rectangular es el factor de rampas $k_{trap}/k_{rect}<1$, que es exactamente cuánto menor sale $D_0$ respecto del ajuste trapezoidal.

In [ ]:
k_rows = []
for (sequence_type, td_ms), curve in data.groupby(['type', 'td_ms'], sort=True):
    tn_ms, n_value, x_ms, ramp_ms = timing(curve)
    positive = curve['g'] > 0
    k_table = float(np.median(curve.loc[positive, 'bvalue'] / curve.loc[positive, 'g']**2))
    k_rows.append({'model': MODEL_NAME, 'type': sequence_type, 'td_ms': td_ms, 'x_ms': x_ms, 'ramp_ms': ramp_ms,
                   'k_model': k_model(tn_ms, n_value, x_ms, ramp_ms), 'k_from_bvalue': k_table,
                   'k_from_bvalue_over_k_model': k_table / k_model(tn_ms, n_value, x_ms, ramp_ms)})
k_check = pd.DataFrame(k_rows)
k_check.to_csv(TABLE_DIR / 'k_check.csv', index=False)
display(k_check)

## Comparación con el otro notebook

Si el otro notebook ya se ejecutó, se comparan los $D_0$ de las señales y del contraste curva por curva (`D0_this / D0_other`).

In [ ]:
other_tables = REPO_ROOT / 'fittings' / OTHER_MODEL_NAME / 'tables'
keys = {'signal_fits.csv': ['type', 'td_ms', 'direction'], 'contrast_fits.csv': ['td_ms', 'direction']}
for name, on in keys.items():
    other_path = other_tables / name
    if not other_path.exists():
        print(f'{other_path} no existe todavía: ejecutar el notebook {OTHER_MODEL_NAME}.')
        continue
    this = pd.read_csv(TABLE_DIR / name, dtype={'direction': str})
    other = pd.read_csv(other_path, dtype={'direction': str})
    merged = this[on + ['D0_mm2_s']].merge(other[on + ['D0_mm2_s']], on=on, suffixes=(f'_{MODEL_NAME}', f'_{OTHER_MODEL_NAME}'))
    merged['ratio_this_over_other'] = merged[f'D0_mm2_s_{MODEL_NAME}'] / merged[f'D0_mm2_s_{OTHER_MODEL_NAME}']
    print(name)
    display(merged)
print('Figuras y tablas guardadas en:', OUTPUT_ROOT)